In [1]:
!pip install torchao --upgrade -q
!pip install transformers peft datasets evaluate scikit-learn accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00


In [2]:
import transformers, peft, torch
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("PyTorch:", torch.__version__)

Transformers: 5.0.0
PEFT: 0.19.1
PyTorch: 2.10.0+cu128


In [3]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import DataLoader
from torch.optim import AdamW
from sklearn.metrics import matthews_corrcoef
import time

# Seed everything for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("PyTorch version:", torch.__version__)

import transformers, peft
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)

Device: cuda
PyTorch version: 2.10.0+cu128
Transformers: 5.0.0
PEFT: 0.19.1


In [4]:
dataset = load_dataset("glue", "cola")

tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/deberta-v3-base",
    use_fast=False
)

def tokenize(examples):
    return tokenizer(
        examples['sentence'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

tokenized = dataset.map(tokenize, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# Sanity check
sample = tokenized['train'][0]
print("input_ids:", sample['input_ids'].shape, sample['input_ids'].dtype)
print("attention_mask:", sample['attention_mask'].shape)
print("labels:", sample['labels'])
print("Train size:", len(tokenized['train']))
print("Val size:", len(tokenized['validation']))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

cola/train-00000-of-00001.parquet:   0%|          | 0.00/251k [00:00<?, ?B/s]

cola/validation-00000-of-00001.parquet:   0%|          | 0.00/37.6k [00:00<?, ?B/s]

cola/test-00000-of-00001.parquet:   0%|          | 0.00/37.7k [00:00<?, ?B/s]

...

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Map:   0%|          | 0/8551 [00:00<?, ? examples/s]

Map:   0%|          | 0/1043 [00:00<?, ? examples/s]

Map:   0%|          | 0/1063 [00:00<?, ? examples/s]

input_ids: torch.Size([128]) torch.int64
attention_mask: torch.Size([128])
labels: tensor(1)
Train size: 8551
Val size: 1043


In [5]:
base_model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-base",
    num_labels=2,
    ignore_mismatched_sizes=True,
    torch_dtype=torch.float32   # CRITICAL - force float32 explicitly
)

# Verify model dtype
param_dtypes = set(p.dtype for p in base_model.parameters())
print("Model parameter dtypes:", param_dtypes)

# Test ONE forward pass before touching LoRA
base_model = base_model.to(device)
base_model.eval()

sample_ids = tokenized['train'][:2]['input_ids'].to(device)
sample_mask = tokenized['train'][:2]['attention_mask'].to(device)
sample_labels = tokenized['train'][:2]['labels'].to(device)

with torch.no_grad():
    test_out = base_model(
        input_ids=sample_ids,
        attention_mask=sample_mask,
        labels=sample_labels
    )

print("Test loss:", test_out.loss.item())
print("Test logits:", test_out.logits)
print("Any NaN in logits?", torch.isnan(test_out.logits).any().item())

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight      

Model parameter dtypes: {torch.float32}
Test loss: 0.6897065043449402
Test logits: tensor([[ 0.1365, -0.0414],
        [-0.0548,  0.1558]], device='cuda:0')
Any NaN in logits? False


In [6]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    target_modules=["query_proj", "value_proj"],
    lora_dropout=0.1,
    bias="none"
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# Verify LoRA applied correctly
lora_layer_count = sum(
    1 for name, mod in model.named_modules()
    if hasattr(mod, 'lora_A')
)
print(f"LoRA applied to {lora_layer_count} layers")

# Test forward pass again after LoRA
model.eval()
with torch.no_grad():
    test_out = model(
        input_ids=sample_ids,
        attention_mask=sample_mask,
        labels=sample_labels
    )
print("Post-LoRA test loss:", test_out.loss.item())
print("Any NaN after LoRA?", torch.isnan(test_out.logits).any().item())

trainable params: 296,450 || all params: 184,720,132 || trainable%: 0.1605
LoRA applied to 24 layers
Post-LoRA test loss: 0.6897065043449402
Any NaN after LoRA? False


In [7]:
train_loader = DataLoader(tokenized['train'], batch_size=16, shuffle=True)
val_loader = DataLoader(tokenized['validation'], batch_size=16)

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-5,            # lower than before
    weight_decay=0.01,
    eps=1e-6            # more numerically stable than default 1e-8
)

total_steps = len(train_loader) * 5
warmup_steps = int(0.1 * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

results_log = {
    "epoch": [], "train_loss": [],
    "val_mcc": [], "epoch_time_sec": []
}

for epoch in range(5):
    model.train()
    total_loss = 0
    valid_batches = 0
    nan_count = 0
    start_time = time.time()

    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss

        if torch.isnan(loss) or torch.isinf(loss):
            nan_count += 1
            if batch_idx < 5:  # print first few NaNs for diagnosis
                print(f"  NaN at batch {batch_idx}, logits: {outputs.logits[:2]}")
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        valid_batches += 1

    # Validation
    model.eval()
    all_preds, all_labels_list = [], []

    with torch.no_grad():
        for batch in val_loader:
            outputs = model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device)
            )
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels_list.extend(batch['labels'].numpy())

    mcc = matthews_corrcoef(all_labels_list, all_preds)
    epoch_time = time.time() - start_time
    avg_loss = total_loss / valid_batches if valid_batches > 0 else float('nan')

    results_log["epoch"].append(epoch + 1)
    results_log["train_loss"].append(round(avg_loss, 4))
    results_log["val_mcc"].append(round(mcc, 4))
    results_log["epoch_time_sec"].append(round(epoch_time, 1))

    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | MCC: {mcc:.4f} | "
          f"Time: {epoch_time:.1f}s | Valid batches: {valid_batches}/{len(train_loader)}")

print("\n=== LoRA Results ===")
print(f"Best MCC: {max(results_log['val_mcc']):.4f}")
print(f"Avg epoch time: {np.mean(results_log['epoch_time_sec']):.1f}s")
print(results_log)
import json

# Save results to file so they survive runtime resets
with open('/content/lora_results.json', 'w') as f:
    json.dump(results_log, f)
print("Results saved to /content/lora_results.json")

# Also print clearly for copy-paste backup
print("\n=== COPY THESE RESULTS ===")
for k, v in results_log.items():
    print(f"{k}: {v}")

Epoch 1 | Loss: 0.6344 | MCC: 0.0000 | Time: 170.3s | Valid batches: 535/535
Epoch 2 | Loss: 0.6093 | MCC: 0.0000 | Time: 174.7s | Valid batches: 535/535
Epoch 3 | Loss: 0.6087 | MCC: 0.0000 | Time: 174.1s | Valid batches: 535/535
Epoch 4 | Loss: 0.6070 | MCC: 0.0000 | Time: 174.2s | Valid batches: 535/535
Epoch 5 | Loss: 0.6078 | MCC: 0.0000 | Time: 174.5s | Valid batches: 535/535

=== LoRA Results ===
Best MCC: 0.0000
Avg epoch time: 173.6s
{'epoch': [1, 2, 3, 4, 5], 'train_loss': [0.6344, 0.6093, 0.6087, 0.607, 0.6078], 'val_mcc': [0.0, 0.0, 0.0, 0.0, 0.0], 'epoch_time_sec': [170.3, 174.7, 174.1, 174.2, 174.5]}
Results saved to /content/lora_results.json

=== COPY THESE RESULTS ===
epoch: [1, 2, 3, 4, 5]
train_loss: [0.6344, 0.6093, 0.6087, 0.607, 0.6078]
val_mcc: [0.0, 0.0, 0.0, 0.0, 0.0]
epoch_time_sec: [170.3, 174.7, 174.1, 174.2, 174.5]


In [8]:
# DIAGNOSTIC: Check what the model is actually predicting
model.eval()
all_preds = []
all_labels_list = []
all_logits = []

with torch.no_grad():
    for batch in val_loader:
        outputs = model(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device)
        )
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels_list.extend(batch['labels'].numpy())
        all_logits.extend(outputs.logits.cpu().numpy())

all_preds = np.array(all_preds)
all_labels_arr = np.array(all_labels_list)

print("=== PREDICTION DISTRIBUTION ===")
print(f"Predicted class 0: {(all_preds == 0).sum()} ({(all_preds == 0).mean()*100:.1f}%)")
print(f"Predicted class 1: {(all_preds == 1).sum()} ({(all_preds == 1).mean()*100:.1f}%)")
print(f"\nActual class 0: {(all_labels_arr == 0).sum()} ({(all_labels_arr == 0).mean()*100:.1f}%)")
print(f"Actual class 1: {(all_labels_arr == 1).sum()} ({(all_labels_arr == 1).mean()*100:.1f}%)")

import numpy as np
logits_arr = np.array(all_logits)
print(f"\nLogits stats:")
print(f"Class 0 logit - mean: {logits_arr[:,0].mean():.4f}, std: {logits_arr[:,0].std():.4f}")
print(f"Class 1 logit - mean: {logits_arr[:,1].mean():.4f}, std: {logits_arr[:,1].std():.4f}")

=== PREDICTION DISTRIBUTION ===
Predicted class 0: 0 (0.0%)
Predicted class 1: 1043 (100.0%)

Actual class 0: 322 (30.9%)
Actual class 1: 721 (69.1%)

Logits stats:
Class 0 logit - mean: -0.6007, std: 0.0388
Class 1 logit - mean: 0.3208, std: 0.0767


In [9]:
# Cell 8 — Retrain LoRA with class weights

# Step 1: Reload fresh base model and re-apply LoRA
base_model_v2 = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-base",
    num_labels=2,
    ignore_mismatched_sizes=True,
    torch_dtype=torch.float32
).to(device)

model_v2 = get_peft_model(base_model_v2, lora_config).to(device)

# Step 2: Compute class weights from training labels
labels_all = np.array(tokenized['train']['labels'])
class_counts = np.bincount(labels_all)
total = len(labels_all)
class_weights = torch.tensor(
    [total / (2 * class_counts[0]),   # class 0 gets higher weight (~1.62)
     total / (2 * class_counts[1])],  # class 1 gets lower weight (~0.72)
    dtype=torch.float32
).to(device)
print("Class weights:", class_weights)
# Expect: class 0 weight > class 1 weight since class 0 is minority

# Step 3: Weighted loss function
import torch.nn as nn
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

# Step 4: Fresh optimizer and scheduler
optimizer_v2 = AdamW(
    filter(lambda p: p.requires_grad, model_v2.parameters()),
    lr=5e-5,
    weight_decay=0.01,
    eps=1e-6
)

total_steps = len(train_loader) * 5
warmup_steps = int(0.1 * total_steps)
scheduler_v2 = get_linear_schedule_with_warmup(
    optimizer_v2,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# Step 5: Training loop with weighted loss
results_lora = {
    "epoch": [], "train_loss": [],
    "val_mcc": [], "epoch_time_sec": []
}

for epoch in range(5):
    model_v2.train()
    total_loss = 0
    valid_batches = 0
    nan_count = 0
    start_time = time.time()

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer_v2.zero_grad()

        outputs = model_v2(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        # Use weighted loss instead of model's default loss
        loss = loss_fn(outputs.logits, labels)

        if torch.isnan(loss) or torch.isinf(loss):
            nan_count += 1
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_v2.parameters(), max_norm=1.0)
        optimizer_v2.step()
        scheduler_v2.step()

        total_loss += loss.item()
        valid_batches += 1

    model_v2.eval()
    all_preds, all_labels_list = [], []

    with torch.no_grad():
        for batch in val_loader:
            outputs = model_v2(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device)
            )
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels_list.extend(batch['labels'].numpy())

    mcc = matthews_corrcoef(all_labels_list, all_preds)
    epoch_time = time.time() - start_time
    avg_loss = total_loss / valid_batches if valid_batches > 0 else float('nan')

    results_lora["epoch"].append(epoch + 1)
    results_lora["train_loss"].append(round(avg_loss, 4))
    results_lora["val_mcc"].append(round(mcc, 4))
    results_lora["epoch_time_sec"].append(round(epoch_time, 1))

    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | MCC: {mcc:.4f} | "
          f"Time: {epoch_time:.1f}s | NaN batches: {nan_count}")

# Save immediately
import json
with open('/content/lora_results_final.json', 'w') as f:
    json.dump(results_lora, f)

print(f"\n=== LORA FINAL RESULTS ===")
print(f"Best MCC: {max(results_lora['val_mcc']):.4f}")
print(f"Avg epoch time: {np.mean(results_lora['epoch_time_sec']):.1f}s")
print(results_lora)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight      

Class weights: tensor([1.6913, 0.7099], device='cuda:0')
Epoch 1 | Loss: 0.6981 | MCC: 0.0485 | Time: 175.3s | NaN batches: 0
Epoch 2 | Loss: 0.6955 | MCC: 0.1015 | Time: 173.9s | NaN batches: 0
Epoch 3 | Loss: 0.6935 | MCC: 0.0487 | Time: 173.6s | NaN batches: 0
Epoch 4 | Loss: 0.6945 | MCC: 0.0799 | Time: 173.2s | NaN batches: 0
Epoch 5 | Loss: 0.6921 | MCC: 0.0998 | Time: 172.7s | NaN batches: 0

=== LORA FINAL RESULTS ===
Best MCC: 0.1015
Avg epoch time: 173.7s
{'epoch': [1, 2, 3, 4, 5], 'train_loss': [0.6981, 0.6955, 0.6935, 0.6945, 0.6921], 'val_mcc': [np.float64(0.0485), np.float64(0.1015), np.float64(0.0487), np.float64(0.0799), np.float64(0.0998)], 'epoch_time_sec': [175.3, 173.9, 173.6, 173.2, 172.7]}


In [10]:
# Cell 9 — LoRA Final Attempt with higher LR and more target modules

# Fresh base model
base_model_v3 = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-base",
    num_labels=2,
    ignore_mismatched_sizes=True,
    torch_dtype=torch.float32
).to(device)

# Expanded target modules
lora_config_v3 = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    target_modules=["query_proj", "key_proj", "value_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none"
)

model_v3 = get_peft_model(base_model_v3, lora_config_v3).to(device)
model_v3.print_trainable_parameters()

# Verify clean start
model_v3.eval()
with torch.no_grad():
    test_out = model_v3(input_ids=sample_ids, attention_mask=sample_mask, labels=sample_labels)
print("Pre-train test loss:", test_out.loss.item())
print("Any NaN?", torch.isnan(test_out.logits).any().item())

# Same class weights as before
loss_fn_v3 = nn.CrossEntropyLoss(weight=class_weights)  # class_weights already on device from Cell 8

# Higher LR — safe now with all stabilizers
optimizer_v3 = AdamW(
    filter(lambda p: p.requires_grad, model_v3.parameters()),
    lr=1e-4,           # bumped from 5e-5
    weight_decay=0.01,
    eps=1e-6
)

n_epochs_v3 = 8
total_steps_v3 = len(train_loader) * n_epochs_v3
warmup_steps_v3 = int(0.1 * total_steps_v3)

scheduler_v3 = get_linear_schedule_with_warmup(
    optimizer_v3,
    num_warmup_steps=warmup_steps_v3,
    num_training_steps=total_steps_v3
)

results_lora_final = {
    "epoch": [], "train_loss": [],
    "val_mcc": [], "epoch_time_sec": []
}

for epoch in range(n_epochs_v3):
    model_v3.train()
    total_loss = 0
    valid_batches = 0
    nan_count = 0
    start_time = time.time()

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer_v3.zero_grad()

        outputs = model_v3(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        loss = loss_fn_v3(outputs.logits, labels)

        if torch.isnan(loss) or torch.isinf(loss):
            nan_count += 1
            if nan_count <= 3:
                print(f"  NaN at epoch {epoch+1}, logits: {outputs.logits[:2]}")
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_v3.parameters(), max_norm=1.0)
        optimizer_v3.step()
        scheduler_v3.step()

        total_loss += loss.item()
        valid_batches += 1

    model_v3.eval()
    all_preds, all_labels_list = [], []

    with torch.no_grad():
        for batch in val_loader:
            outputs = model_v3(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device)
            )
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels_list.extend(batch['labels'].numpy())

    mcc = matthews_corrcoef(all_labels_list, all_preds)
    epoch_time = time.time() - start_time
    avg_loss = total_loss / valid_batches if valid_batches > 0 else float('nan')

    results_lora_final["epoch"].append(epoch + 1)
    results_lora_final["train_loss"].append(round(avg_loss, 4))
    results_lora_final["val_mcc"].append(round(mcc, 4))
    results_lora_final["epoch_time_sec"].append(round(epoch_time, 1))

    print(f"Epoch {epoch+1}/{n_epochs_v3} | Loss: {avg_loss:.4f} | MCC: {mcc:.4f} | "
          f"Time: {epoch_time:.1f}s | NaN: {nan_count}")

# Save immediately
with open('/content/lora_results_final_v3.json', 'w') as f:
    json.dump(results_lora_final, f, default=str)

print(f"\n=== LORA FINAL RESULTS ===")
print(f"Best MCC: {max(results_lora_final['val_mcc']):.4f}")
print(f"Avg epoch time: {np.mean(results_lora_final['epoch_time_sec']):.1f}s")
print(results_lora_final)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight      

trainable params: 443,906 || all params: 184,867,588 || trainable%: 0.2401
Pre-train test loss: 0.5798882246017456
Any NaN? False
Epoch 1/8 | Loss: 0.6978 | MCC: 0.0045 | Time: 172.3s | NaN: 0
Epoch 2/8 | Loss: 0.6962 | MCC: -0.0284 | Time: 180.0s | NaN: 0
Epoch 3/8 | Loss: 0.6936 | MCC: 0.0682 | Time: 182.4s | NaN: 0
Epoch 4/8 | Loss: 0.6879 | MCC: 0.1243 | Time: 182.6s | NaN: 0
Epoch 5/8 | Loss: 0.6806 | MCC: 0.1257 | Time: 182.4s | NaN: 0
Epoch 6/8 | Loss: 0.6776 | MCC: 0.1557 | Time: 182.5s | NaN: 0
Epoch 7/8 | Loss: 0.6671 | MCC: 0.1489 | Time: 182.8s | NaN: 0
Epoch 8/8 | Loss: 0.6690 | MCC: 0.1395 | Time: 182.8s | NaN: 0

=== LORA FINAL RESULTS ===
Best MCC: 0.1557
Avg epoch time: 181.0s
{'epoch': [1, 2, 3, 4, 5, 6, 7, 8], 'train_loss': [0.6978, 0.6962, 0.6936, 0.6879, 0.6806, 0.6776, 0.6671, 0.669], 'val_mcc': [np.float64(0.0045), np.float64(-0.0284), np.float64(0.0682), np.float64(0.1243), np.float64(0.1257), np.float64(0.1557), np.float64(0.1489), np.float64(0.1395)], 'epoch_t

In [11]:
# Cell 10 — Compute LoRA Effective Rank
import numpy as np

def compute_effective_rank(matrix):
    with torch.no_grad():
        matrix_cpu = matrix.float().cpu()
        try:
            _, singular_values, _ = torch.linalg.svd(matrix_cpu, full_matrices=False)
        except Exception:
            matrix_np = matrix_cpu.numpy()
            _, singular_values_np, _ = np.linalg.svd(matrix_np, full_matrices=False)
            singular_values = torch.tensor(singular_values_np)

        singular_values = singular_values.float()
        singular_values = singular_values[singular_values > 1e-10]

        if len(singular_values) == 0:
            return 0.0

        p = singular_values / singular_values.sum()
        entropy = -torch.sum(p * torch.log(p))
        return torch.exp(entropy).item()

print("=== LoRA Effective Rank per Layer ===")
eranks = []
layer_names = []

for name, module in model_v3.named_modules():
    if hasattr(module, 'lora_A') and hasattr(module, 'lora_B'):
        for key in module.lora_A:
            A = module.lora_A[key].weight
            B = module.lora_B[key].weight
            combined = B @ A
            erank = compute_effective_rank(combined)
            eranks.append(erank)
            layer_names.append(name)
            print(f"{name} | Shape: {combined.shape} | Effective rank: {erank:.3f}")

print(f"\n=== SUMMARY ===")
print(f"Mean effective rank: {np.mean(eranks):.3f}")
print(f"Min effective rank:  {np.min(eranks):.3f}")
print(f"Max effective rank:  {np.max(eranks):.3f}")
print(f"Max possible rank (r=8): 8")
print(f"Total LoRA layers measured: {len(eranks)}")

# Save
lora_erank_summary = {
    "mean_erank": round(float(np.mean(eranks)), 3),
    "min_erank": round(float(np.min(eranks)), 3),
    "max_erank": round(float(np.max(eranks)), 3),
    "per_layer": {name: round(er, 3) for name, er in zip(layer_names, eranks)}
}

import json
with open('/content/lora_erank.json', 'w') as f:
    json.dump(lora_erank_summary, f)
print("\nEffective rank saved to /content/lora_erank.json")

=== LoRA Effective Rank per Layer ===
base_model.model.deberta.encoder.layer.0.attention.self.query_proj | Shape: torch.Size([768, 768]) | Effective rank: 7.593
base_model.model.deberta.encoder.layer.0.attention.self.key_proj | Shape: torch.Size([768, 768]) | Effective rank: 7.481
base_model.model.deberta.encoder.layer.0.attention.self.value_proj | Shape: torch.Size([768, 768]) | Effective rank: 7.723
base_model.model.deberta.encoder.layer.1.attention.self.query_proj | Shape: torch.Size([768, 768]) | Effective rank: 7.690
base_model.model.deberta.encoder.layer.1.attention.self.key_proj | Shape: torch.Size([768, 768]) | Effective rank: 7.775
base_model.model.deberta.encoder.layer.1.attention.self.value_proj | Shape: torch.Size([768, 768]) | Effective rank: 7.742
base_model.model.deberta.encoder.layer.2.attention.self.query_proj | Shape: torch.Size([768, 768]) | Effective rank: 7.797
base_model.model.deberta.encoder.layer.2.attention.self.key_proj | Shape: torch.Size([768, 768]) | Effect